Score commercial auto policies for expected claim risk
Fits a model on historical data and scores the current period

Data:
train.csv  - policies 2020-2022, includes claim outcomes
score.csv  - policies 2023-2024, no outcomes (to be scored)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import FunctionTransformer, TargetEncoder

In [ ]:
# ---- load ----
df_training_data = pd.read_csv("../data/train.csv")


print("train:", df_training_data.shape)


In [ ]:
# ---- preprocessing ----

# fill numeric missing with 0 -- good enough for now
df_training_data[df_training_data.select_dtypes("number").columns] = df_training_data.select_dtypes("number").fillna(0)


# encode categoricals for train
df_training_data["coverage_type"] = df_training_data["coverage_type"].astype("category").cat.codes
df_training_data["business_type"]  = df_training_data["business_type"].astype("category").cat.codes
df_training_data["state"]          = df_training_data["state"].astype("category").cat.codes



In [ ]:
# ---- target ----
# using binary had-a-claim flag as target
df_training_data["target"] = (df_training_data["claim_count"] > 0).astype(int)
print("positive rate:", round(df_training_data["target"].mean(), 3))

In [ ]:
# ---- features ----
feat_cols = [
    "coverage_type",
    "vehicle_count", "vehicle_avg_age", "driver_count", "driver_avg_age",
    "years_in_business", "prior_year_mileage_000",
    "business_type", "state",
    "prior_apd_claim_count", "prior_al_claim_count", "prior_loss_amount",
    "deductible", "coverage_limit_000", "annual_premium",
    "risk_score_external", "num_heavy_vehicles", "late_payment_count",
    "claim_paid_amount_current_period",
    "days_to_first_claim_report",
]

X = df_training_data[feat_cols]
y = df_training_data["target"]

In [ ]:
# ---- fit model ----

# random split -- not time-based
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

tmp = LogisticRegression(max_iter=1000)
tmp.fit(X_train, y_train)

In [ ]:
# ---- evaluate ----
x1 = tmp.predict(X_train)
print("accuracy:", accuracy_score(y_train, x1))
# results look solid

In [ ]:
df_data_to_score = pd.read_csv("../data/score.csv")
print("score:", df_data_to_score.shape)
df_data_to_score[df_data_to_score.select_dtypes("number").columns] = df_data_to_score.select_dtypes("number").fillna(0)

# same for score
df_data_to_score["coverage_type"] = df_data_to_score["coverage_type"].astype("category").cat.codes
df_data_to_score["business_type"]  = df_data_to_score["business_type"].astype("category").cat.codes
df_data_to_score["state"]          = df_data_to_score["state"].astype("category").cat.codes

In [ ]:
# ---- score new policies ----
X2 = df_data_to_score[feat_cols]
preds = tmp.predict(X2)

df_data_to_score["risk_score"] = preds
df_data_to_score[["policy_id", "risk_score"]].to_csv("predictions.csv")
print("done -- predictions.csv written")